# non_sport_crypto (main) daily data demonstration

This notebook is the supervisor-facing demonstration layer.

It reads only the prepared files in data/demo/:

- main_market_manifest.csv;
- main_market_metadata.csv;
- main_daily_candles.csv.gz;
- export_summary.json.

There are no SQL queries and no SQLite dependency in this notebook.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "data/demo/export_summary.json").exists()
    ),
    Path.cwd(),
)
DATA_DIR = PROJECT_ROOT / "data" / "demo"

MANIFEST_PATH = DATA_DIR / "main_market_manifest.csv"
CANDLES_PATH = DATA_DIR / "main_daily_candles.csv.gz"
SUMMARY_PATH = DATA_DIR / "export_summary.json"

with SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    export_summary = json.load(handle)

manifest = pd.read_csv(MANIFEST_PATH)
candles = pd.read_csv(CANDLES_PATH, compression="gzip")

for column in ["open_time", "close_time"]:
    manifest[column] = pd.to_datetime(manifest[column], utc=True, errors="coerce")

candles["date_utc"] = pd.to_datetime(candles["date_utc"], utc=True, errors="coerce")
candles["end_period_utc"] = pd.to_datetime(
    candles["end_period_ts"], unit="s", utc=True, errors="coerce"
)

date_columns = {"open_time", "close_time", "date_utc", "end_period_utc"}
text_columns = {
    "market_ticker", "event_ticker", "series_ticker",
    "title", "result", "status", "market_status",
}
for frame in [manifest, candles]:
    for column in frame.columns:
        if column not in date_columns and column not in text_columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")

print("Snapshot status:", export_summary["run_status"])
print("Candle run:", export_summary["candle_run_id"])
print("Manifest rows:", len(manifest))
print("Candle rows:", len(candles))

In [ ]:
METADATA_PATH = DATA_DIR / "main_market_metadata.csv"
market_metadata = pd.read_csv(METADATA_PATH)
manifest["market_id"] = pd.read_csv(
    MANIFEST_PATH, usecols=["market_id"], dtype="string"
)["market_id"]
candles["market_id"] = pd.read_csv(
    CANDLES_PATH, usecols=["market_id"], dtype="string"
)["market_id"]

for column in ["open_time", "close_time"]:
    market_metadata[column] = pd.to_datetime(
        market_metadata[column], utc=True, errors="coerce"
    )

metadata_fields = [
    "market_question", "market_subtitle", "yes_subtitle",
    "no_subtitle", "market_rules", "market_result",
    "event_question", "event_subtitle",
    "series_title", "series_subtitle",
    "series_category", "series_frequency",
]
metadata_lookup = market_metadata[
    ["market_id", *metadata_fields]
].copy()
if not metadata_lookup["market_id"].is_unique:
    raise ValueError("Market metadata must contain one row per market_id.")

manifest = manifest.merge(
    metadata_lookup,
    on="market_id",
    how="left",
    validate="one_to_one",
)
candles = candles.merge(
    metadata_lookup,
    on="market_id",
    how="left",
    validate="many_to_one",
)
manifest["title"] = manifest["market_question"].fillna(manifest["title"])

print("Market metadata rows:", len(market_metadata))
print("Markets without a question:", int(manifest["market_question"].isna().sum()))
display(Markdown("### Question and market context"))
display(market_metadata[[
    "market_id", "market_question", "market_subtitle",
    "event_question", "series_title", "market_rules",
]].head(10))

## 1. What was downloaded

This is a compact inventory of the prepared snapshot and its time range.

In [ ]:
overview = pd.DataFrame([
    {"metric": "markets in manifest", "value": manifest["market_ticker"].nunique()},
    {"metric": "markets with candles", "value": candles["market_ticker"].nunique()},
    {"metric": "events represented", "value": manifest["event_ticker"].nunique()},
    {"metric": "series represented", "value": manifest["series_ticker"].nunique()},
    {"metric": "daily candle rows", "value": len(candles)},
    {"metric": "first UTC date", "value": candles["date_utc"].min()},
    {"metric": "last UTC date", "value": candles["date_utc"].max()},
    {"metric": "markets with zero exported candles", "value": int(
        manifest["actual_candles_exported"].eq(0).sum()
    )},
])
display(overview)

if export_summary["run_status"] != "success":
    display(Markdown(
        f"Snapshot status: {export_summary['run_status']}. "
        "The source run was not fully successful; this status is intentionally visible."
    ))

## 2. Selection and daily representation

The manifest keeps both filter decisions. The active download used the union cohort, so a market entered when it passed at least one filter.

In [ ]:
filter_overlap = (
    manifest.groupby(
        ["passes_volume_filter", "passes_lifetime_filter", "passes_both_filters"],
        dropna=False,
    )
    .agg(
        markets=("market_ticker", "nunique"),
        mean_volume_fp=("volume_fp", "mean"),
        mean_lifetime_days=("lifetime_days", "mean"),
        exported_candles=("actual_candles_exported", "sum"),
    )
    .reset_index()
)
display(filter_overlap.round(2))

daily_representation = (
    candles.groupby("date_utc")
    .agg(
        markets=("market_ticker", "nunique"),
        candle_rows=("market_ticker", "size"),
        mean_close=("price_close", "mean"),
        median_close=("price_close", "median"),
    )
    .reset_index()
    .sort_values("date_utc")
)

display(Markdown("### Dates with the highest market representation"))
display(daily_representation.sort_values(
    ["markets", "candle_rows"], ascending=False
).head(15))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
daily_representation.plot(
    x="date_utc", y="markets", ax=axes[0], color="tab:blue",
    title="Distinct markets represented by UTC date", legend=False
)
daily_representation.plot(
    x="date_utc", y="candle_rows", ax=axes[1], color="tab:orange",
    title="Daily candle rows by UTC date", legend=False
)
axes[0].set_ylabel("markets")
axes[1].set_ylabel("candle rows")
axes[1].set_xlabel("UTC date")
plt.tight_layout()
plt.show()

monthly_representation = (
    daily_representation.assign(
        month_utc=daily_representation["date_utc"].dt.strftime("%Y-%m")
    )
    .groupby("month_utc")
    .agg(
        represented_dates=("date_utc", "nunique"),
        peak_markets=("markets", "max"),
        total_candle_rows=("candle_rows", "sum"),
        mean_markets_per_date=("markets", "mean"),
    )
    .reset_index()
)
display(monthly_representation)

## 3. Example daily probability paths

These examples use the raw daily closing probability. They are descriptive only; no outcome alignment or Brier Score is calculated here.

In [ ]:
sample_tickers = (
    manifest.sort_values(
        ["volume_fp", "actual_candles_exported"],
        ascending=False,
    )["market_ticker"]
    .head(3)
    .tolist()
)

fig, axes = plt.subplots(
    len(sample_tickers), 1,
    figsize=(14, 3.5 * len(sample_tickers)),
    squeeze=False,
)

for row_index, ticker in enumerate(sample_tickers):
    sample = candles.loc[candles["market_ticker"].eq(ticker)].sort_values("date_utc")
    title = manifest.loc[
        manifest["market_ticker"].eq(ticker), "title"
    ].dropna().head(1)
    title_text = title.iloc[0] if len(title) else ticker

    axis = axes[row_index, 0]
    axis.plot(sample["date_utc"], sample["price_close"], color="tab:blue", linewidth=1.5)
    axis.set_ylim(0, 1)
    axis.set_ylabel("probability")
    axis.set_title(f"{ticker}: {title_text}")
    axis.grid(True, alpha=0.3)

axes[-1, 0].set_xlabel("UTC date")
plt.tight_layout()
plt.show()

display(manifest.loc[
    manifest["market_ticker"].isin(sample_tickers),
    [
        "market_ticker", "title", "volume_fp", "lifetime_days",
        "actual_candles_exported", "result",
    ],
])

## 4. Basic data-quality view

These are descriptive checks on the prepared files, not corrections to the raw data.

In [ ]:
quality = pd.DataFrame([{
    "duplicate_market_day_rows": int(
        candles.duplicated(["market_ticker", "end_period_ts"]).sum()
    ),
    "missing_close_probability": int(candles["price_close"].isna().sum()),
    "close_below_zero": int(candles["price_close"].lt(0).sum()),
    "close_above_one": int(candles["price_close"].gt(1).sum()),
    "missing_bid_close": int(candles["yes_bid_close"].isna().sum()),
    "missing_ask_close": int(candles["yes_ask_close"].isna().sum()),
    "non_daily_intervals": int(candles["period_interval"].ne(1440).sum()),
}])
display(quality)

display(Markdown(
    "The raw daily layer is sufficient for inspecting probability paths, "
    "market coverage, volume, open interest, and quote availability. "
    "The next research layer must define the forecast timestamp, align "
    "each market with its resolved outcome, and calculate Brier Score."
))